In [1]:
install.packages("sqldf")

install.packages("dplyr")

install.packages("ggplot2")

install.packages("readr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [2]:
library(sqldf)

library(dplyr)

library(ggplot2)

library(readr)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
unzip("/content/northstar_dataset.zip",
      exdir="/content/northstar_data")

print("Dataset Extracted Successfully")

[1] "Dataset Extracted Successfully"


In [4]:
list.files("/content/northstar_data/northstar_dataset")

[1] "app_events.csv"      "complaints.csv"      "customers.csv"      
 [4] "data_dictionary.csv" "deliveries.csv"      "drivers.csv"        
 [7] "hubs.csv"            "incidents.csv"       "orders.csv"         
[10] "README.txt"          "vehicles.csv"

In [5]:
deliveries <- read_csv(
"/content/northstar_data/northstar_dataset/deliveries.csv"
)

orders <- read_csv(
"/content/northstar_data/northstar_dataset/orders.csv"
)

complaints <- read_csv(
"/content/northstar_data/northstar_dataset/complaints.csv"
)

customers <- read_csv(
"/content/northstar_data/northstar_dataset/customers.csv"
)

print("Datasets Loaded Successfully")

Rows: 950 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (6): delivery_id, order_id, driver_id, vehicle_id, hub_id, delivery_status
dbl  (5): route_distance_km, manual_route_override_count, proof_of_completio...
dttm (2): dispatch_time, delivery_completed_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1250 Columns: 11
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): order_id, customer_id, service_type, pickup_zone, dropoff_zone, pr...
dbl  (3): promised_window_hours, order_value, special_handling_flag
dttm (1): order_created_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 320 Columns: 10
── Column specification ─────────────────────────

[1] "Datasets Loaded Successfully"


In [6]:
result1 <- sqldf("
SELECT delivery_status,
COUNT(*) AS total
FROM deliveries
GROUP BY delivery_status
")

print(result1)

  delivery_status total
1         Delayed   202
2          Failed   132
3          OnTime   616


## Delivery Status Analysis

The SQL analysis shows the distribution of delivery outcomes across the organisation. A significant number of delayed and failed deliveries indicates operational inefficiencies in dispatch coordination, routing, or logistics management. These delivery issues may contribute directly to customer dissatisfaction and increased complaint levels.

In [7]:
result2 <- sqldf("
SELECT hub_id,
COUNT(*) AS failed_deliveries
FROM deliveries
WHERE delivery_status = 'Failed'
GROUP BY hub_id
ORDER BY failed_deliveries DESC
")

print(result2)

  hub_id failed_deliveries
1    H08                26
2    H05                23
3    H01                17
4    H04                16
5    H06                15
6    H07                14
7    H03                11
8    H02                10


In [8]:
result3 <- sqldf("
SELECT complaint_type,
COUNT(*) AS total_complaints
FROM complaints
GROUP BY complaint_type
ORDER BY total_complaints DESC
")

print(result3)

     complaint_type total_complaints
1             Delay              101
2      MissedPickup               64
3          AppIssue               53
4   DriverBehaviour               51
5 SupportExperience               20
6           Billing               16
7            Damage               15


In [9]:
result4 <- sqldf("
SELECT
o.service_type,
d.delivery_status,
COUNT(*) AS total
FROM orders o
JOIN deliveries d
ON o.order_id = d.order_id
GROUP BY o.service_type, d.delivery_status
")

print(result4)

   service_type delivery_status total
1      Business         Delayed    28
2      Business          Failed    25
3      Business          OnTime    73
4       Medical         Delayed    22
5       Medical          Failed    16
6       Medical          OnTime    70
7        Parcel         Delayed    49
8        Parcel          Failed    25
9        Parcel          OnTime   156
10    Passenger         Delayed    53
11    Passenger          Failed    38
12    Passenger          OnTime   171
13       Retail         Delayed    50
14       Retail          Failed    28
15       Retail          OnTime   146


In [10]:
failed_deliveries <- deliveries %>%
filter(delivery_status == 'Failed')

result5 <- sqldf("
SELECT *
FROM failed_deliveries
")

print(head(result5))

  delivery_id order_id driver_id vehicle_id hub_id       dispatch_time
1     DL00001   O00938      D004       V056    H05 2024-06-18 10:57:00
2     DL00010   O00836      D058       V057    H08 2025-09-22 19:09:00
3     DL00012   O01207      D051       V017    H05 2024-12-26 19:41:00
4     DL00022   O01027      D088       V011    H07 2025-08-24 00:21:00
5     DL00026   O00906      D092       V055    H04 2025-02-04 11:16:00
6     DL00033   O00885      D041       V075    H06 2024-11-17 14:01:00
  delivery_completed_at delivery_status route_distance_km
1   2024-06-19 09:05:59          Failed             17.26
2   2025-09-23 01:15:29          Failed              9.85
3   2024-12-27 09:26:05          Failed             16.96
4   2025-08-24 04:25:39          Failed             15.81
5   2025-02-06 01:48:45          Failed             14.27
6   2024-11-19 06:51:25          Failed             11.72
  manual_route_override_count proof_of_completion_missing
1                           1          

## Query Optimisation Insight

The query performance was improved by pre-filtering failed delivery records before executing the SQL operation. This reduces the number of rows processed during querying and improves execution efficiency. Optimisation techniques are important for scalable analytics and high-volume operational systems.